# Image processing notebook: From overlap corrected to transmission 

### 00 - exp 1XX acquisition 00

##  Initial settings

### Import libraries
Import all the required libraries

In [1]:
import sys
sys.path.append(r'..\01_Functions')
from step_functions import *
from dict_functions import *
from proc_functions import *
from img_functions import *
%matplotlib inline

### Select directories
Select the source directory. This directory is where the images **after** the overlap correction were saved.
Select the destination directory. Here is where the transmission images are going to be saved.

In [2]:
# %load select_directory('src_dir')
src_dir = r"J:\900 Varia\2021\000_tony_data\03_Processed_step_by_step\old_testing\testing_overlap_corrected_sa\exp1XX"

In [3]:
# %load select_directory('dst_dir')
dst_dir = r"J:\900 Varia\2021\000_tony_data\03_Processed_step_by_step\01_New_transmission_results\exp1XX_SA_testing190722"

### Selec folders to process

In [4]:
stack_dict = prep_stack_dict(src_dir)
for key in stack_dict.keys():
    print(key)

00_ob
01_so_ref
02_sa
02_sa_2_2
03_ob_end


In [5]:
ref_folder = ['01_so_ref']
proc_folder = ['02_sa']

## TEST processing for reference images

### Build the reference images dictionary 
keep_acq_numb: amount of acquisition folders to process. this is a test so we want to so it fast. thus, we limit the amount of data <br>
In case you want to get an image _(array, header)_ in the dictionary, follow the format:<br>

`test = ref_test_dict['01_so_ref'][0][10]`<br>
"variable" = "dictionary name" ["key/folder to process"]["number of aquisition folder"]["image number"]

**_To visualize the image_** (a tuple with an array in position 0 and a header in position 1, you require the next instruction:<br>

`show_img(test[0])
test[1]` # to show the header of that image

In [ ]:
ref_test_dict, ref_test_param = reading (src_dir, proc_folder = ref_folder, keep_acq_numb = 4)

### Pre-processing sequence with parameters
The pre-processing sequence refers to any sequence with their respective parameters that emphasize on modifying or enhancing the image (**usually filters**). In this step you can also perform an averaging of all the acquisitions `stack_averaging`, a step binning of the aquisitions `binning_acquisitions`, and/or a step binning of the frames `binning_frames`.

**The image processing (SBKG, registration, scrubbing, and/or TFC should be done in the processing step.**

**NOTE: The order of the functions in the sequence matters**, but not the order of the parameters, you can also include alien parameters, this will not affect the process.

In [ ]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(ref_test_param,['threshold'], [0])

In [ ]:
ref_test_dict = pre_processing_step (ref_test_dict, pre_proc_seq, param_dict = ref_test_param)

### Processing sequence and variables to obtain the reference TFC image

In [ ]:
proc_seq = [scrubbing_correction_dict, SBKG_correction_dict]

In [ ]:
BB_mask = get_img(src_dir + '/bb_mask_ref_full.fits')
add_to_dict(ref_test_param,['BB_mask'], [BB_mask])

In [ ]:
ref_test_dict = processing_step (ref_test_dict, proc_seq, param_dict = ref_test_param)

### Extract image for the TFC 

In [ ]:
ref_img_TFC = avg_frames_dict (ref_test_dict[ref_folder[0]], output_type = 'img')

### Get NCA
There will be displacements in the experiment images, for this reason, it is better to get the `nca` from the reference image.

In [ ]:
# %load select_rois(ref_img_TFC[0], list_rois = ['nca'])
nca = [406, 398, 47, 12]

In [ ]:
show_img_rois(ref_img_TFC[0], dr = [(nca, 'blue')])

In [ ]:
add_to_dict(ref_test_param,['nca'], [nca])

## TEST processing for experiment images

In [ ]:
exp_test_dict, exp_test_param = testing_mode_step (src_dir, proc_folder = proc_folder, keep_acq_numb= 4)

The pre processing sequence will not change. It will be the same as the one we used for the reference. Except if there is some averaging requirements

In [ ]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(exp_test_param,['threshold'], [0])

In [ ]:
exp_test_dict = pre_processing_step (exp_test_dict, pre_proc_seq, param_dict = exp_test_param)

### Displacement correction
For this specific experiment, there is just a y-axis displacement. Therefore, `dof = ['ty']`

We also knew previously the ROIs of the regions for the correction `reg_rois_list`. Otherwise, we would have to select them in an extra step

In [ ]:
reg_img = get_img(src_dir + '/reg_img_LE_full.fits')
reg_rois_list = [([24, 350, 30, 135], 'v'), ([432, 350, 30, 135], 'v')]

img = exp_test_dict[proc_folder[0]][0][50]
show_img_rois(img[0], dr = [(reg_rois_list, 'yellow')])

In [ ]:
img_reg_corr, M = img_registration (img, reg_img, reg_rois_list = reg_rois_list, dof=['ty'])
#show_img(img_reg_corr[0], cmap = 'gray')
print(M)

In [ ]:
add_to_dict(exp_test_param,['reg_img', 'reg_rois_list', 'M', 'dof', 'ref_dict'], 
            [reg_img, reg_rois_list, M, ['ty'], ref_test_dict])

### Processing sequence and variables (EXP)

In [ ]:
proc_seq = [scrubbing_correction_dict, image_registration_dict, SBKG_correction_dict, TFC_correction_dict]

In [ ]:
BB_mask = get_img(src_dir + '/bb_mask_full.fits')
add_to_dict(exp_test_param,['BB_mask', 'nca', 'use_ref'], [BB_mask, nca, True])

In [ ]:
exp_test_dict = processing_step (exp_test_dict, proc_seq, param_dict = exp_test_param)

## Reference AVG Processing

In [6]:
ref_dict, ref_param = reading_step (src_dir, proc_folder = ref_folder)

Reading Images: 100%|████████████████████████████| 3/3 [01:10<00:00, 23.45s/it]


In [36]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(ref_param,['threshold'], [0])

In [37]:
ref_dict = pre_processing_step (ref_dict, pre_proc_seq, param_dict = ref_param)

Processing Filetring : 100%|█████████████████████| 6/6 [02:34<00:00, 25.67s/it]


In [38]:
proc_seq = [scrubbing_correction_dict, SBKG_correction_dict]

In [39]:
BB_mask = get_img(src_dir + '/bb_mask_ref_full.fits')
add_to_dict(ref_param,['BB_mask'], [BB_mask])

In [40]:
ref_dict = processing_step (ref_dict, proc_seq, param_dict = ref_param)

Processing SBKG Correction: 100%|████████████████| 1/1 [00:18<00:00, 18.74s/it]


In [41]:
dst_dir_test = dst_dir + '/SO_avg_noTFC'
saving_step (ref_dict, dst_dir_test, img_name = 'SO_avg_noTFC')

Saving images as a single acquisition


Saving Images: 100%|█████████████████████████████| 1/1 [00:21<00:00, 21.03s/it]


In [42]:
proc_seq_save_ref = [TFC_correction_dict]
param_save_ref = {}
add_to_dict(param_save_ref,['nca', 'use_ref', 'ref_dict'], [nca, False, ref_dict])
ref_dict_save_ref = processing_step (ref_dict, proc_seq = proc_seq_save_ref, param_dict = param_save_ref)

Processing TFC Correction: 100%|█████████████████| 1/1 [00:00<00:00,  4.90it/s]


In [43]:
dst_dir_test = dst_dir + '/SO_avg_test'
saving_step (ref_dict, dst_dir_test, img_name = 'SO_avg_testing')

Saving images as a single acquisition


Saving Images: 100%|█████████████████████████████| 1/1 [00:21<00:00, 21.05s/it]


## Experiment AVG processing

In [15]:
exp_dict, exp_param = reading_step (src_dir, proc_folder = proc_folder)

Reading Images: 100%|████████████████████████████| 3/3 [00:20<00:00,  6.67s/it]


In [16]:
pre_proc_seq = [outlier_removal, stack_averaging]
add_to_dict(exp_param,['threshold'], [0])

In [17]:
exp_dict = pre_processing_step (exp_dict, pre_proc_seq, param_dict = exp_param)

Processing Filetring : 100%|█████████████████████| 6/6 [02:58<00:00, 29.79s/it]


In [18]:
disp = float(-0.1999999)
M = np.array([[1,0,disp], [0,1,0], [0,0,1]])
nca = [406, 398, 47, 12]

In [19]:
proc_seq = [scrubbing_correction_dict, image_registration_dict, SBKG_correction_dict, TFC_correction_dict]

In [21]:
BB_mask = get_img(src_dir + '/bb_mask_full.fits')
add_to_dict(exp_param,['BB_mask', 'nca', 'use_ref', 'M', 'dof', 'ref_dict'], 
                        [BB_mask, nca, False, M, ['ty'], ref_dict])

In [22]:
exp_dict = processing_step (exp_dict, proc_seq, param_dict = exp_param)

Processing TFC Correction: 100%|█████████████████| 1/1 [00:00<00:00,  4.95it/s]


In [24]:
dst_dir_test = dst_dir + '/SA_avg_test'
saving_step (exp_dict, dst_dir_test, img_name = 'SA_avg_testing')

Saving images as a single acquisition


Saving Images: 100%|█████████████████████████████| 1/1 [00:21<00:00, 21.13s/it]
